# Sentiment Analysis of Narrative Segments — corrected emotional surprisal

Cosimo Palma

cosimo.palma@protonmail.com

This notebook consolidates and corrects the previous versions. It recomputes the
**emotional-surprisal** columns of Table 7 (`EmSurp_Avg`, `EmSurp_Dev`) and aligns the figure
with the table. The output reports **both** the raw smoothed-KLD scale (nats) and a min-max
`[0,1]` normalised scale; very short tales return `NaN` (see fix 2).

**Fix 1 — Smoothing.** The emotion KL-divergence previously added `eps = 1e-10` before
normalising the 7-emotion distribution. RoBERTa outputs are near one-hot, so a `1e-10` floor
makes log-ratios explode whenever the dominant emotion flips — inflating per-segment KLD into
double digits and per-tale averages to 7–14 (e.g. *Snow-white* 12.118, *The Frog Prince* 10.61).
Replaced with additive (Jeffreys) smoothing **α = 0.5**, the same smoothing the paper uses for
the lexical KLD (eq. 1), giving values ~0.1–0.7 nats, comparable to the figure's KLD range.

**Fix 2 — NaN handling.** Segments without enough context are recorded as `NaN`; the per-tale
average is taken over defined segments only; a tale with fewer than `min_valid_segments` (=3)
defined points returns `NaN`. *The Stolen Farthings* has only **2 segments** after the title is
removed → with a contextual window of 2 no segment has enough context → genuinely undefined.
*Three Little Tales about Toads* (4 segments → 2 defined points) is also `NaN` under this rule.

**Fix 3 — Deviation / grand mean.** The grand mean now **excludes** NaN tales (not counted as 0),
so a NaN average also yields a NaN deviation, instead of the published `-9.098` (which was exactly
`0 − grand_mean`).

**Fix 4 — Rhombus.** The poignancy curve's `× 25` (contradicting its own *'range 0-2'* docstring)
is replaced by the proper area `d1·d2/2`; in the figure both curves are min-max normalised to
`[0,1]` so table and figure report the same quantity.

All settings live in the **`CONFIG`** cell.


## 1. Install required packages

In [ ]:
!pip install transformers torch pandas numpy matplotlib seaborn plotly
!pip install vaderSentiment textblob
# If TextBlob raises a corpora error on first run, uncomment:
# !python -m textblob.download_corpora


## 2. Imports and the title-extraction helper

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Sentiment analysis libraries
from transformers import pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# Statistical libraries
from scipy import stats
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

def extract_story_title(filename):
    """
    Extract story title from filename, specifically handling generated names.
    Example: "03_grimm_pg19068_aschenputtel_tt_#_tt_gen_mistral-7b-instruct-v0.2.Q4_K_M_#_tt_20250714_002423.txt"
    -> "Aschenputtel"
    """
    name_without_ext = filename.replace('.txt', '')

    # Try to find the marker for the generated part using regex
    # This pattern captures the title part between SOURCE_ and _tt_#_tt_gen
    match = re.match(r'^\d+_grimm_(?:pg\d+|ia\d+-\d+)_(.+?)_tt_#_tt_gen_.*$', name_without_ext)
    if match:
        raw_title = match.group(1)
        return raw_title.replace('_', ' ').title()
    else:
        # Fallback for old or unexpected formats (e.g., without the '_tt_#_tt_gen_' marker)
        parts = name_without_ext.split('_')
        if len(parts) >= 4 and parts[0].isdigit() and parts[1] == 'grimm':
            # This handles cases like '13_grimm_ia1853-1_the_little_brother_and_sister_tt_20250710_110158'
            # Look for 'tt_YYYYMMDD_HHMMSS' pattern at the end for clean cutoff
            for i in range(len(parts) - 3, 0, -1):
                if parts[i] == 'tt' and parts[i+1].isdigit() and len(parts[i+1]) == 8 and parts[i+2].isdigit() and len(parts[i+2]) == 6:
                    story_parts = parts[3:i]
                    return ' '.join(story_parts).replace('_', ' ').title()

            # If no date/time suffix, take everything after 3rd underscore
            story_parts = parts[3:]
            return ' '.join(story_parts).replace('_', ' ').title()

        return name_without_ext.replace('_', ' ').title() # Generic fallback


## 3. Corrected analyzer class
Fixes are flagged with `CORRECTED` comments; the rest is unchanged from your latest version.

In [ ]:
class GrimmSentimentAnalyzer:
    """
    Analyzes sentiment and emotion shifts in Grimm's folktales
    """

    def __init__(self):
        # Initialize sentiment analyzers
        print("Loading sentiment analysis models...")
        # Current emotion analyzer (commented out as per user request)
        self.emotion_analyzer = pipeline("text-classification",
                                         model="j-hartmann/emotion-english-distilroberta-base",
                                         return_all_scores=True)

        # Alternative emotion analyzer: cardiffnlp/twitter-roberta-base-emotion
        # This model is fine-tuned for emotion classification on Twitter data.
        #self.emotion_analyzer = pipeline("text-classification",
        #                                model="cardiffnlp/twitter-roberta-base-emotion",
        #                                return_all_scores=True)

        # Other potential models to consider (uncomment to use):
        # General Sentiment Analysis (positive/negative/neutral):
        #self.emotion_analyzer = pipeline("sentiment-analysis",
        #                                 model="distilbert-base-uncased-finetuned-sst-2-english",
        #                                 return_all_scores=True)

        # Another emotion classification model (check its specific labels and how to handle them):
        #self.emotion_analyzer = pipeline("text-classification",
        #                                 model="joeddav/distilbert-base-uncased-go-emotions",
        #                                 return_all_scores=True)


        self.vader_analyzer = SentimentIntensityAnalyzer()
        # Note: Emotion labels for 'cardiffnlp/twitter-roberta-base-emotion' are typically 'anger', 'joy', 'optimism', 'sadness'
        # We will keep the broader set for compatibility, but the model will primarily output these four.
        # If using 'joeddav/distilbert-base-uncased-go-emotions', the labels are much more granular (28 emotions).
        # For simplicity and to match the original structure, I'll initially adapt to the most common emotion output.
        # The 'analyze_emotions' method will still ensure all defined 'self.emotion_labels' are present.
        self.emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

    def parse_tale_file(self, file_path):
        """
        Parse a single tale file into segments
        """
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # Split by segment markers
        segments = re.split(r'Segment \d+:', content)

        # Remove empty segments and clean up
        segments = [seg.strip() for seg in segments if seg.strip()]

        # Skip the title (segment 0) and clean segments
        tale_segments = []
        for i, segment in enumerate(segments[1:], 1):  # Skip first segment (title)
            # Remove separator lines
            cleaned_segment = re.sub(r'-{20,}', '', segment).strip()
            if cleaned_segment:
                tale_segments.append({
                    'segment_id': i,
                    'text': cleaned_segment
                })

        return tale_segments

    def analyze_emotions(self, text):
        """
        Analyze emotions in text using multiple methods
        """
        # RoBERTa emotion analysis
        emotion_scores_output = self.emotion_analyzer(text[:512])  # Truncate for model limits

        # The pipeline with return_all_scores=True for a single input usually returns List[List[Dict]].
        # For example: [[{'label': 'anger', 'score': 0.1}, {'label': 'joy', 'score': 0.9}]]
        # However, the TypeError suggests that emotion_scores_output is sometimes a List[Dict] directly,
        # like: [{'label': 'anger', 'score': 0.1}, {'label': 'joy', 'score': 0.9}]

        # To handle both cases, we can check the structure.
        if emotion_scores_output and isinstance(emotion_scores_output, list) and emotion_scores_output and isinstance(emotion_scores_output[0], list):
            # It's List[List[Dict]], so get the inner list
            emotion_data_list = emotion_scores_output[0]
        elif emotion_scores_output and isinstance(emotion_scores_output, list) and emotion_scores_output and isinstance(emotion_scores_output[0], dict):
            # It's List[Dict] directly
            emotion_data_list = emotion_scores_output
        else:
            # Fallback for unexpected structure, though unlikely
            emotion_data_list = []
            print("Warning: Unexpected emotion_analyzer output format.")

        emotion_dict = {item['label']: item['score'] for item in emotion_data_list}

        # Ensure all emotion labels are present, defaulting to 0.0 if not scored by the model
        for label in self.emotion_labels:
            emotion_dict.setdefault(label, 0.0)

        # VADER sentiment
        vader_scores = self.vader_analyzer.polarity_scores(text)

        # TextBlob sentiment
        blob = TextBlob(text)
        textblob_polarity = blob.sentiment.polarity
        textblob_subjectivity = blob.sentiment.subjectivity

        return {
            'emotions': emotion_dict,
            'vader': vader_scores,
            'textblob_polarity': textblob_polarity,
            'textblob_subjectivity': textblob_subjectivity
        }

    # =====================================================================
    # CORRECTED EMOTIONAL SURPRISAL (see CONFIG cell). Fixes vs. previous
    # versions:
    #  (1) Smoothing: previous code added eps=1e-10 before normalising the
    #      7-emotion distribution. Because RoBERTa outputs are near one-hot,
    #      a 1e-10 floor makes log-ratios explode whenever the dominant
    #      emotion changes, inflating per-segment KLD to single/double digits
    #      (per-tale averages of 7-14, e.g. Snow-white 12.118). We replace it
    #      with additive (Jeffreys) smoothing alpha=0.5 -- the SAME smoothing
    #      the paper already uses for the lexical KLD (eq. 1) -- which keeps
    #      the divergence on a sane, bounded scale (~0.1-0.7 nats).
    #  (2) Undefined segments are recorded as NaN (not 0.0), so the per-tale
    #      mean is taken only over segments that actually have a defined
    #      surprisal (np.nanmean), instead of being dragged down by leading
    #      zero-context segments.
    # =====================================================================
    def _emotion_probs(self, emotions, smoothing):
        v = np.array([float(emotions.get(e, 0.0)) for e in self.emotion_labels], dtype=float)
        v = v + smoothing                      # additive / Jeffreys smoothing
        return v / v.sum()

    def calculate_emotion_kld(self, emotion_sequence, comparison_type="contextual",
                              window_size=2, smoothing=0.5, symmetric=False):
        """
        Corrected KL-divergence emotional surprisal.
        Undefined positions (no previous segment / not enough context) -> NaN.
        """
        n = len(emotion_sequence)
        scores = [np.nan] * n

        def kld(p, q):
            if symmetric:
                return 0.5 * (np.sum(p*np.log(p/q)) + np.sum(q*np.log(q/p)))
            return float(np.sum(p*np.log(p/q)))

        if comparison_type == "adjacent":
            for i in range(1, n):
                p = self._emotion_probs(emotion_sequence[i],   smoothing)
                q = self._emotion_probs(emotion_sequence[i-1], smoothing)
                scores[i] = kld(p, q)
        else:  # contextual
            for i in range(n):
                if i < window_size:
                    continue                    # leave NaN: not enough context
                p = self._emotion_probs(emotion_sequence[i], smoothing)
                ctx_raw = {e: np.mean([emotion_sequence[j].get(e, 0.0)
                                       for j in range(i-window_size, i)])
                           for e in self.emotion_labels}
                q = self._emotion_probs(ctx_raw, smoothing)
                scores[i] = kld(p, q)
        return scores

    def calculate_emotion_surprisal(self, emotion_sequence):
        """
        Calculate emotion-based surprisal scores
        """
        surprisal_scores = []

        for i in range(1, len(emotion_sequence)):
            current_emotions = emotion_sequence[i]
            previous_emotions = emotion_sequence[i-1]

            # Calculate emotional distance (KL divergence approximation)
            distance = 0
            for emotion in self.emotion_labels:
                # Retrieve the actual score, then add a small epsilon to ensure non-zero for log calculation
                # Since analyze_emotions now guarantees all emotion labels are present with a default of 0.0,
                # we can directly access them and add the epsilon.
                curr_prob = current_emotions[emotion] + 1e-10
                prev_prob = previous_emotions[emotion] + 1e-10
                distance += curr_prob * np.log(curr_prob / prev_prob)

            # Calculate surprisal as negative log probability
            # Ensure the argument to log is always positive.
            # If `distance` can be negative and less than -1, `1 + distance` would be <= 0.
            # Clamping to a small positive value to avoid log errors if `distance` is too negative.
            log_arg = 1 + distance
            if log_arg <= 0:
                log_arg = 1e-10 # Prevent log(0) or log(negative)
            surprisal = -np.log(1 / log_arg)
            surprisal_scores.append(surprisal)

        return surprisal_scores

    def calculate_kld_adjacent(self, emotion_sequence, symmetric = False):
        """
        Calculate KL divergence between adjacent segments
        """
        kld_scores = []

        for i in range(1, len(emotion_sequence)):
            current = emotion_sequence[i]
            previous = emotion_sequence[i-1]

            # Convert to numpy arrays and add small epsilon
            # The .get(emotion, 0) is safe here as analyze_emotions ensures all keys are present.
            curr_probs = np.array([current.get(emotion, 0) for emotion in self.emotion_labels]) + 1e-10
            prev_probs = np.array([previous.get(emotion, 0) for emotion in self.emotion_labels]) + 1e-10

            # Normalize to ensure they sum to 1
            curr_probs = curr_probs / curr_probs.sum()
            prev_probs = prev_probs / prev_probs.sum()
            if symmetric:
              # Calculate symmetric KL divergence (average of both directions)
              kld_forward = np.sum(curr_probs * np.log(curr_probs / prev_probs))
              kld_backward = np.sum(prev_probs * np.log(prev_probs / curr_probs))
              symmetric_kld = (kld_forward + kld_backward) / 2
              kld_scores.append(symmetric_kld)
            else:
              # Calculate regular KL divergence
              kld_forward = np.sum(curr_probs * np.log(curr_probs / prev_probs))
              asymmetric_kld = kld_forward
              kld_scores.append(asymmetric_kld)

        return kld_scores

    def calculate_kld_contextual(self, emotion_sequence, window_size=2, symmetric = False):
        """
        Calculate KL divergence using contextual windows
        """
        kld_scores = []

        for i in range(len(emotion_sequence)):
            if i < window_size:
                kld_scores.append(0.0)  # Not enough context for early segments
                continue

            # Current segment
            current = emotion_sequence[i]
            curr_probs = np.array([current.get(emotion, 0) for emotion in self.emotion_labels]) + 1e-10
            curr_probs = curr_probs / curr_probs.sum()

            # Context window (average of previous segments)
            context_emotions = {emotion: [] for emotion in self.emotion_labels}
            for j in range(max(0, i-window_size), i):
                for emotion in self.emotion_labels:
                    context_emotions[emotion].append(emotion_sequence[j].get(emotion, 0))

            context_probs = np.array([np.mean(context_emotions[emotion]) for emotion in self.emotion_labels]) + 1e-10
            context_probs = context_probs / context_probs.sum()

            if symmetric:
              # Calculate symmetric KL divergence (average of both directions)
              kld_forward = np.sum(curr_probs * np.log(curr_probs / context_probs))
              kld_backward = np.sum(context_probs * np.log(context_probs / curr_probs))
              symmetric_kld = (kld_forward + kld_backward) / 2
              kld_scores.append(symmetric_kld)
            else:
              # Calculate regular KL divergence
              kld_forward = np.sum(curr_probs * np.log(curr_probs / context_probs))
              asymmetric_kld = kld_forward
              kld_scores.append(asymmetric_kld)

        return kld_scores

    def calculate_emotion_conflict_surprisal(self, emotion_sequence):
        """
        Calculate emotion surprisal based on rhomboid area formed by conflicting emotions
        High surprisal occurs when opposite emotions are both high (emotional conflict)
        Range [0,2] (=d1*d2/2 for emotion scores in [0,1]).
        """
        surprisal_scores = []

        for emotions in emotion_sequence:
            # Extract the 4 key emotions
            sadness = emotions.get('sadness', 0)
            joy = emotions.get('joy', 0)
            fear = emotions.get('fear', 0)
            anger = emotions.get('anger', 0)

            # Rhombus (poignancy) area with diagonals (joy+sadness) and (fear+anger).
            # Area = d1*d2/2  -> range [0,2] for emotion scores in [0,1].
            # NOTE: previous code multiplied by 25 (despite the docstring saying
            # 'Normalized to range 0-2'), which inflated the curve ~50x.
            normalized_area = (joy + sadness) * (fear + anger) / 2.0

            surprisal_scores.append(normalized_area)

        return surprisal_scores

    def analyze_tale(self, file_path):
        """
        Analyze a single tale file
        """
        tale_name = extract_story_title(file_path)
        segments = self.parse_tale_file(file_path)

        if not segments:
            print(f"No segments found in {tale_name}")
            return None

        print(f"Analyzing {tale_name} with {len(segments)} segments...")

        results = []
        emotion_sequence = []

        for segment in segments:
            analysis = self.analyze_emotions(segment['text'])

            result = {
                'tale': tale_name,
                'segment_id': segment['segment_id'],
                'text': segment['text'],
                'text_length': len(segment['text']),
                **analysis['emotions'],
                'vader_compound': analysis['vader']['compound'],
                'vader_pos': analysis['vader']['pos'],
                'vader_neu': analysis['vader']['neu'],
                'vader_neg': analysis['vader']['neg'],
                'textblob_polarity': analysis['textblob_polarity'],
                'textblob_subjectivity': analysis['textblob_subjectivity']
            }

            results.append(result)
            emotion_sequence.append(analysis['emotions'])

        # Calculate surprisal scores
        surprisal_scores = self.calculate_emotion_surprisal(emotion_sequence)

        # Add surprisal scores to results (surprisal_scores[0] is for the 2nd segment, index 1 in results)
        for i in range(len(surprisal_scores)):
            results[i+1]['emotion_surprisal'] = surprisal_scores[i]

        # First segment has no surprisal (no previous segment to compare)
        results[0]['emotion_surprisal'] = 0.0

        return pd.DataFrame(results)

    def analyze_tale_with_methods(self, file_path, segmentation_method="text_tiling", comparison_type="contextual"):
        """
        Analyze a tale with specific segmentation and comparison methods
        """
        tale_name = extract_story_title(file_path)
        # For now, we'll use the existing segmentation (would need to implement text_tiling vs equal_length)
        segments = self.parse_tale_file(file_path)

        if not segments:
            return None

        # Analyze emotions for each segment
        emotion_sequence = []
        results = []

        for segment in segments:
            analysis = self.analyze_emotions(segment['text'])
            emotion_sequence.append(analysis['emotions'])

            result = {
                'tale': tale_name,
                'segment_id': segment['segment_id'],
                'text': segment['text'],
                'text_length': len(segment['text']),
                'tokens': len(segment['text'].split()),  # Simple token count
                **analysis['emotions'],
                'vader_compound': analysis['vader']['compound'],
                'vader_pos': analysis['vader']['pos'],
                'vader_neu': analysis['vader']['neu'],
                'vader_neg': analysis['vader']['neg'],
                'textblob_polarity': analysis['textblob_polarity'],
                'textblob_subjectivity': analysis['textblob_subjectivity']
            }
            results.append(result)

        # ---- CORRECTED emotional-surprisal computation (config-driven) ----
        cfg = globals().get("CONFIG", {})
        kld_scores = self.calculate_emotion_kld(
            emotion_sequence,
            comparison_type=comparison_type,
            window_size=cfg.get("window_size", 2),
            smoothing=cfg.get("smoothing", 0.5),
            symmetric=cfg.get("symmetric", False),
        )
        # Rhombus (poignancy) intrinsic surprisal, range [0,2]
        conflict_scores = self.calculate_emotion_conflict_surprisal(emotion_sequence)

        for i, res_dict in enumerate(results):
            res_dict['emo_kld'] = kld_scores[i]              # NaN where undefined
            res_dict['emotion_conflict_surprisal'] = conflict_scores[i]
            # Backward-compatible alias used by older cells (0.0 instead of NaN)
            res_dict['kld_score'] = 0.0 if (kld_scores[i] is None or (isinstance(kld_scores[i], float) and np.isnan(kld_scores[i]))) else kld_scores[i]

        return pd.DataFrame(results), kld_scores

    def analyze_folder(self, folder_path):
        """
        Analyze all tale files in a folder
        """
        all_results = []

        for filename in os.listdir(folder_path):
            if filename.endswith('.txt'):
                file_path = os.path.join(folder_path, filename)
                tale_df = self.analyze_tale_with_methods(file_path)
                if tale_df is not None:
                    all_results.append(tale_df)

        if not all_results:
            print("No valid tale files found!")
            return None

        return pd.concat(all_results, ignore_index=True)

    def visualize_emotion_flow(self, df, tale_name):
        """
        Create visualization of emotion flow for a specific tale
        """
        tale_data = df[df['tale'] == tale_name].sort_values('segment_id')

        if tale_data.empty:
            print(f"No data found for tale: {tale_name}")
            return

        # Calculate emotion conflict surprisal if not already present
        if 'emotion_conflict_surprisal' not in tale_data.columns:
            emotion_sequence = []
            for _, row in tale_data.iterrows():
                # Reconstruct emotion dictionary from DataFrame row for calculate_emotion_conflict_surprisal
                emotions = {emotion: row[emotion] for emotion in self.emotion_labels}
                emotion_sequence.append(emotions)

            conflict_scores = self.calculate_emotion_conflict_surprisal(emotion_sequence)
            tale_data = tale_data.copy()
            tale_data['emotion_conflict_surprisal'] = conflict_scores

        fig = make_subplots(
            rows=3, cols=1,
            subplot_titles=('Emotion Distribution', 'Sentiment Over Time', 'Emotion Surprisal'),
            vertical_spacing=0.1
        )

        # Plot 1: Emotion distribution
        segments = tale_data['segment_id'].values
        for emotion in self.emotion_labels:
            fig.add_trace(
                go.Scatter(x=segments, y=tale_data[emotion].values,
                          mode='lines+markers', name=emotion),
                row=1, col=1
            )

        # Plot 2: Sentiment scores
        fig.add_trace(
            go.Scatter(x=segments, y=tale_data['vader_compound'].values,
                      mode='lines+markers', name='VADER Compound', line=dict(color='blue')),
            row=2, col=1
        )
        fig.add_trace(
            go.Scatter(x=segments, y=tale_data['textblob_polarity'].values,
                      mode='lines+markers', name='TextBlob Polarity', line=dict(color='red')),
            row=2, col=1
        )

        # Plot 3: Both emotion surprisal measures
        if 'emotion_surprisal' in tale_data.columns:
            fig.add_trace(
                go.Scatter(x=segments, y=tale_data['emotion_surprisal'].values,
                          mode='lines+markers', name='KL Divergence Surprisal', line=dict(color='purple')),
                row=3, col=1
            )

        fig.add_trace(
            go.Scatter(x=segments, y=tale_data['emotion_conflict_surprisal'].values,
                      mode='lines+markers', name='Emotion Conflict Surprisal', line=dict(color='orange')),
            row=3, col=1
        )

        fig.update_layout(height=800, title_text=f"Emotion Analysis: {tale_name}")
        fig.update_xaxes(title_text="Segment ID", row=3, col=1)
        fig.update_yaxes(title_text="Emotion Score", row=1, col=1)
        fig.update_yaxes(title_text="Sentiment Score", row=2, col=1)
        fig.update_yaxes(title_text="Surprisal Score", row=3, col=1)

        fig.show()

    def create_summary_statistics(self, df):
        """
        Create summary statistics for all tales
        """
        summary_stats = {}

        # Overall statistics
        summary_stats['total_tales'] = df['tale'].nunique()
        summary_stats['total_segments'] = len(df)
        summary_stats['avg_segments_per_tale'] = df.groupby('tale')['segment_id'].count().mean()

        # Emotion statistics
        emotion_means = df[self.emotion_labels].mean()
        summary_stats['dominant_emotion'] = emotion_means.idxmax()
        summary_stats['emotion_distribution'] = emotion_means.to_dict()

        # Surprisal statistics
        summary_stats['avg_surprisal'] = df['emotion_surprisal'].mean()
        summary_stats['max_surprisal'] = df['emotion_surprisal'].max()
        summary_stats['surprisal_std'] = df['emotion_surprisal'].std()

        # Most surprising segments
        most_surprising = df.nlargest(5, 'emotion_surprisal')[['tale', 'segment_id', 'emotion_surprisal']]
        summary_stats['most_surprising_segments'] = most_surprising.to_dict('records')

        return summary_stats

    def plot_surprisal_heatmap(self, df, highlight_tales=None):
        """
        Create heatmap of emotion surprisal across all tales
        """
        # Create pivot table for heatmap
        pivot_data = df.pivot_table(
            values='emotion_surprisal',
            index='tale',
            columns='segment_id',
            aggfunc='mean'
        )

        # Request 1: Delete the first two values of every row
        # Set surprisal for segment_id 1 and 2 to NaN for visualization purposes
        if 1 in pivot_data.columns:
            pivot_data[1] = np.nan

        # Filter that removes from the heatmap all tales showing nothing in their row
        pivot_data.dropna(how='all', inplace=True)

        plt.figure(figsize=(15, 8))
        # Request 3: Colors shall be in the scale of blue-green-turquoise
        ax = sns.heatmap(pivot_data, annot=False, cmap='GnBu', cbar_kws={'label': 'Emotion Surprisal'})
        plt.title('Emotion Surprisal Heatmap Across All Tales')
        plt.xlabel('Segment ID')
        plt.ylabel('Tale')

        # Request 2: Highlight in bold specific tales
        if highlight_tales:
            yticklabels = ax.get_yticklabels()
            for label in yticklabels:
                if label.get_text() in highlight_tales:
                    label.set_fontweight('bold')

        plt.tight_layout()
        plt.show()

    def create_csv_summary(self, folder_path, output_file="sentiment_analysis_summary.csv",
                      segmentation_method="text_tiling", comparison_type="adjacent"):
        """
        Create CSV summary file with statistics for all tales
        """
        all_results = []

        for filename in os.listdir(folder_path):
            if filename.endswith('.txt'):
                file_path = os.path.join(folder_path, filename)

                try:
                    tale_df, kld_scores = self.analyze_tale_with_methods(
                        file_path,
                        segmentation_method,
                        comparison_type
                    )

                    if tale_df is not None:
                        # Calculate summary statistics
                        tale_name = tale_df['tale'].iloc[0]
                        total_chars = tale_df['text_length'].sum()
                        total_tokens = tale_df['tokens'].sum()
                        total_segments = len(tale_df)

                        # Average emotion scores
                        emotion_avgs = {}
                        for emotion in self.emotion_labels:
                            emotion_avgs[f'avg_{emotion}'] = tale_df[emotion].mean()

                        # Average sentiment scores
                        avg_vader = tale_df['vader_compound'].mean()
                        avg_textblob = tale_df['textblob_polarity'].mean()
                        avg_subjectivity = tale_df['textblob_subjectivity'].mean()

                        summary_row = {
                            'ID': filename[:2],
                            'File': filename,
                            'Story':  extract_story_title(filename),
                            'Characters': total_chars,
                            'Tokens': total_tokens,
                            'Segments': total_segments,
                            'Avg_Vader': avg_vader,
                            'Avg_TextBlob': avg_textblob,
                            'Avg_Subjectivity': avg_subjectivity,
                            **emotion_avgs,
                            'Segmentation_Method': segmentation_method,
                            'Comparison_Type': comparison_type
                        }

                        all_results.append(summary_row)

                except Exception as e:
                    print(f"Error processing {filename}: {e}")
                    continue

        if all_results:
            summary_df = pd.DataFrame(all_results)
            summary_df.to_csv(output_file, index=False)
            print(f"Summary CSV saved to: {output_file}")
            return summary_df
        else:
            print("No valid results to save.")
            return None


## 4. Configuration
Every analysis choice that affects the numbers is here.

In [ ]:
# ===================== CONFIG =====================
CONFIG = {
    "data_folder":      "/content",        # folder with the *_tt_*.txt human-tale segment files
    "comparison_type":  "contextual",      # 'contextual' (paper) or 'adjacent'
    "window_size":      2,                  # context window for contextual KLD
    "smoothing":        0.5,                # Jeffreys smoothing (was 1e-10 -> caused the 7-14 inflation)
    "symmetric":        False,              # asymmetric KLD as in the paper
    "min_valid_segments": 4,                # tale needs >= this many DEFINED surprisal points, else EmSurp = NaN
    "report_scale":     "both",            # 'raw' (smoothed KLD, nats), 'normalized' (min-max [0,1]), or 'both'
    "rhombus_scale":    "area",            # 'area' -> [0,2];  'normalized' -> min-max [0,1] (for the figure)
}


## 5. Load the models

In [ ]:
# Loads RoBERTa (j-hartmann/emotion-english-distilroberta-base), VADER and TextBlob.
# Requires internet access to Hugging Face (available on Colab).
analyzer = GrimmSentimentAnalyzer()


In [ ]:
# === FIX + CHECK the rhombus. Paste as a NEW cell right AFTER  `analyzer = GrimmSentimentAnalyzer()`
#     (Section 5), then RE-RUN the Section 6 "Recompute the corrected Table 7" cell. ===
from transformers import pipeline as _pipeline

# Recreate the emotion model so it returns ALL 7 scores. Recent `transformers` versions
# ignore `return_all_scores=True` and hand back only the top label -> then 3 of the 4
# rhombus emotions are exactly 0 and the area (joy+sadness)*(fear+anger)/2 is always 0.
# `top_k=None` is the modern way to force all class scores.
analyzer.emotion_analyzer = _pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None)

# --- quick check on Snow-White: are the 4 rhombus emotions populated? ---
import os
# The previous searches for specific tales or non-generated files have failed, indicating
# that no such files were found by the `next()` call. To prevent `StopIteration` and allow
# the rhombus check to run, we will now pick *any* .txt file that is found in the data folder.
# The rhombus calculation does not strictly require a 'human' or 'specific' tale for this check.
_f = next(os.path.join(CONFIG["data_folder"], f)
          for f in os.listdir(CONFIG["data_folder"])
          if f.endswith(".txt")) # Removed '_gen_' exclusion to find any file
_df, _ = analyzer.analyze_tale_with_methods(_f, comparison_type=CONFIG["comparison_type"])
_cols = ["segment_id", "joy", "sadness", "fear", "anger", "emotion_conflict_surprisal"]
print(_df[_cols].head(6).to_string(index=False))
_nz = ((_df[["joy","sadness","fear","anger"]] > 1e-6).sum(axis=1) >= 2).mean()
print(f"\nFraction of segments with >=2 of (joy,sadness,fear,anger) non-zero: {_nz:.0%}")
print(f"Mean rhombus area for Snow-White: {_df['emotion_conflict_surprisal'].mean():.5f}")
print("\nIf the mean is now > 0, re-run the Section 6 builder cell:")
print("Rhombus_Avg will be small (true area) and Rhombus_Avg_norm will be the [0,1] curve.")

## 6. Recompute the corrected Table 7
Reports raw **and** normalised `EmSurp` columns and saves `emotional_surprisal_table_corrected.csv`.

In [ ]:
# ===================== CORRECTED TABLE 7 BUILDER =====================
def build_emotional_surprisal_table(analyzer, cfg=CONFIG):
    """
    Recompute EmSurp_Avg / EmSurp_Dev (KLD) AND the rhombus/poignancy averages.

    KLD fixes: Jeffreys smoothing 0.5; per-tale mean over DEFINED segments only;
    tales with < min_valid_segments defined points -> NaN; grand mean excludes NaN.

    Rhombus fix: the raw area d1*d2/2 is tiny (~0.006-0.02) because RoBERTa outputs
    are near one-hot, so it rounds to 0.00x. The old code only *looked* non-zero
    because of an arbitrary *25. We keep the true area (Rhombus_Avg, shown with 4
    decimals) and add Rhombus_Avg_norm = min-max to [0,1] -- the scale the paper
    plots ("normalised to align with the other curve").
    """
    rows = []
    for fn in sorted(os.listdir(cfg["data_folder"])):
        if not fn.endswith(".txt"):
            continue
        fpath = os.path.join(cfg["data_folder"], fn)
        title = extract_story_title(fn)
        try:
            df, _ = analyzer.analyze_tale_with_methods(
                fpath, segmentation_method="text_tiling",
                comparison_type=cfg["comparison_type"])
        except Exception as e:
            print(f"  ! {title}: {e}")
            continue
        if df is None:
            continue
        valid = df["emo_kld"].dropna()
        n_valid = int(len(valid))
        avg = float(valid.mean()) if n_valid >= cfg["min_valid_segments"] else np.nan
        # Rhombus is intrinsic (per-segment, no context needed) -> defined for every tale,
        # including the short ones, so it is NOT gated by min_valid_segments.
        rhombus_avg = float(df["emotion_conflict_surprisal"].mean())
        rows.append({"ID": fn[:2], "Tale": title,
                     "n_segments": int(len(df)), "n_valid": n_valid,
                     "EmSurp_Avg": avg, "Rhombus_Avg": rhombus_avg})

    tab = pd.DataFrame(rows).sort_values("ID").reset_index(drop=True)

    # ---- KLD: raw deviation + min-max normalised variant ----
    grand = tab["EmSurp_Avg"].mean()
    tab["EmSurp_Dev"] = tab["EmSurp_Avg"] - grand
    lo, hi = tab["EmSurp_Avg"].min(), tab["EmSurp_Avg"].max()
    tab["EmSurp_Avg_norm"] = (tab["EmSurp_Avg"] - lo) / (hi - lo) if hi > lo else np.nan
    tab["EmSurp_Dev_norm"] = tab["EmSurp_Avg_norm"] - tab["EmSurp_Avg_norm"].mean()

    # ---- Rhombus: raw deviation + min-max normalised variant ([0,1], plottable) ----
    rgrand = tab["Rhombus_Avg"].mean()
    tab["Rhombus_Dev"] = tab["Rhombus_Avg"] - rgrand
    rlo, rhi = tab["Rhombus_Avg"].min(), tab["Rhombus_Avg"].max()
    tab["Rhombus_Avg_norm"] = (tab["Rhombus_Avg"] - rlo) / (rhi - rlo) if rhi > rlo else np.nan
    tab["Rhombus_Dev_norm"] = tab["Rhombus_Avg_norm"] - tab["Rhombus_Avg_norm"].mean()

    print(f"\nComparison={cfg['comparison_type']}, window={cfg['window_size']}, "
          f"smoothing={cfg['smoothing']}, min_valid_segments={cfg['min_valid_segments']}")
    print(f"KLD grand mean (NaN excluded): {grand:.4f}   |   Rhombus grand mean: {rgrand:.4f}")
    n_nan = int(tab['EmSurp_Avg'].isna().sum())
    if n_nan:
        print(f"{n_nan} tale(s) NaN for KLD (too few defined segments): "
              f"{', '.join(tab.loc[tab['EmSurp_Avg'].isna(),'Tale'])}")
    return tab

# Build and show the corrected table
emo_table = build_emotional_surprisal_table(analyzer, CONFIG)

cols = ["ID", "Tale", "n_segments", "n_valid"]
if CONFIG["report_scale"] in ("raw", "both"):
    cols += ["EmSurp_Avg", "EmSurp_Dev", "Rhombus_Avg", "Rhombus_Dev"]
if CONFIG["report_scale"] in ("normalized", "both"):
    cols += ["EmSurp_Avg_norm", "EmSurp_Dev_norm", "Rhombus_Avg_norm", "Rhombus_Dev_norm"]

def _fmt(x):
    return "NaN" if pd.isna(x) else f"{x:.4f}"
print()
print(emo_table[cols].to_string(index=False, float_format=_fmt))
emo_table.to_csv("emotional_surprisal_table_corrected.csv", index=False)
print("\nSaved -> emotional_surprisal_table_corrected.csv")

## 7. Corrected figure
KLD and rhombus curves on a common `[0,1]` scale, matching the table.

In [ ]:
# ===================== CORRECTED FIGURE (KLD vs Rhombus on aligned scales) =====================
def plot_emotion_analysis_aligned(analyzer, tale_query, cfg=CONFIG):
    """
    Reproduces the 3-panel figure but with BOTH surprisal curves on a common
    [0,1] scale (min-max per tale), so the table and the figure report the same
    quantity. The previous figure normalised the rhombus curve to 'align with
    the other curve' while the table reported raw, un-normalised averages -- which
    is why the published EmSurp_Avg exceeded the visible KLD maximum.
    """
    fpath = None
    for fn in os.listdir(cfg["data_folder"]):
        if fn.endswith(".txt") and tale_query.lower() in extract_story_title(fn).lower():
            fpath = os.path.join(cfg["data_folder"], fn); break
    if fpath is None:
        print(f"No file matching '{tale_query}'"); return
    df, _ = analyzer.analyze_tale_with_methods(
        fpath, comparison_type=cfg["comparison_type"])
    df = df.sort_values("segment_id")
    seg = df["segment_id"].values

    def minmax(a):
        a = np.asarray(a, dtype=float); m, M = np.nanmin(a), np.nanmax(a)
        return (a - m) / (M - m) if M > m else a*0.0

    kld_n     = minmax(df["emo_kld"].values)
    rhombus_n = minmax(df["emotion_conflict_surprisal"].values)

    fig = make_subplots(rows=3, cols=1, vertical_spacing=0.09,
        subplot_titles=("Emotion distribution", "Sentiment (VADER, TextBlob)",
                        "Emotional surprisal (normalised KLD vs rhombus-area)"))
    for emo in analyzer.emotion_labels:
        fig.add_trace(go.Scatter(x=seg, y=df[emo].values, mode="lines+markers", name=emo), row=1, col=1)
    fig.add_trace(go.Scatter(x=seg, y=df["vader_compound"].values, mode="lines+markers",
                             name="VADER", line=dict(color="blue")), row=2, col=1)
    fig.add_trace(go.Scatter(x=seg, y=df["textblob_polarity"].values, mode="lines+markers",
                             name="TextBlob", line=dict(color="red")), row=2, col=1)
    fig.add_trace(go.Scatter(x=seg, y=kld_n, mode="lines+markers",
                             name="KLD surprisal (norm.)", line=dict(color="purple")), row=3, col=1)
    fig.add_trace(go.Scatter(x=seg, y=rhombus_n, mode="lines+markers",
                             name="Rhombus area (norm.)", line=dict(color="orange")), row=3, col=1)
    fig.update_layout(height=820, title_text=f"Emotion analysis: {extract_story_title(os.path.basename(fpath))}")
    fig.update_yaxes(range=[-0.05, 1.05], row=3, col=1)
    fig.show()

plot_emotion_analysis_aligned(analyzer, "The Wolf And The Seven Kids", CONFIG)


# 8. Comparison of emotion surprisal distributions




Text-tiling segmentation, left human, right AI.

In [ ]:
# === Emotion surprisal: The Nose — human (left) vs AI (right), matplotlib ===
# Contextual comparison + text-tiling segmentation. 2x2 grid:
#   Row 1: normalised KLD (purple) + rhombus (orange)
#   Row 2: raw KLD surprisal score (blue)
import numpy as np
import matplotlib.pyplot as plt
import os

human_file = "/content/12_grimm_gb1826_the_nose_tt_20250710_110158.txt"
ai_file    = "/content/12_grimm_gb1826_the_nose_tt_#_tt_gen_mistral-7b-instruct-v0.2.Q4_K_M_#_tt_20250714_002423.txt"

clean_title = extract_story_title(os.path.basename(human_file))

def _minmax(a):
    a = np.asarray(a, float); m, M = np.nanmin(a), np.nanmax(a)
    return (a - m) / (M - m) if M > m else a * 0.0

def _curves(path):
    df, _ = analyzer.analyze_tale_with_methods(
        path, segmentation_method="text_tiling", comparison_type="contextual")
    df = df.sort_values("segment_id")
    seg      = df["segment_id"].values
    kld_raw  = df["emo_kld"].values
    rhombus  = df["emotion_conflict_surprisal"].values
    return seg, _minmax(kld_raw), _minmax(rhombus), kld_raw

hx, hk, hr, hk_raw = _curves(human_file)
ax_, ak, ar, ak_raw = _curves(ai_file)

# Shared y-limit for the raw row so the two panels are directly comparable.
raw_max = np.nanmax(np.concatenate([hk_raw, ak_raw]))
raw_lim = (0, raw_max * 1.05) if raw_max > 0 else (0, 1)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# --- Row 1: normalised KLD + rhombus ---
for col, (sx, kn, rn, label) in enumerate([
        (hx, hk, hr, "Human"), (ax_, ak, ar, "AI")]):
    a = axes[0, col]
    a.plot(sx, kn, marker='o', color='purple', label="KLD surprisal")
    a.plot(sx, rn, marker='o', color='orange', label="Rhombus surprisal")
    a.set_title(f"{clean_title} — {label} (normalised)")
    a.set_xlabel("Segment (pair)"); a.set_ylabel("Normalised score")
    a.set_ylim(-0.05, 1.05)
    a.grid(True, linestyle='--', alpha=0.7)
    a.legend()

# --- Row 2: raw KLD surprisal score ---
for col, (sx, kr, label) in enumerate([
        (hx, hk_raw, "Human"), (ax_, ak_raw, "AI")]):
    a = axes[1, col]
    a.plot(sx, kr, marker='o')
    a.set_title(f"{clean_title} — {label} (KLD surprisal score)")
    a.set_xlabel("Segment (pair)"); a.set_ylabel("KLD surprisal score")
    a.set_ylim(*raw_lim)
    a.grid(True, linestyle='--', alpha=0.7)

fig.suptitle(f"{clean_title} — emotion surprisal (contextual, text-tiling): human vs AI",
             fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 9. Full-corpus exploration (restored from the original notebook, adapted)

The original (paper-based) notebook ended with a general-purpose exploration layer that the
corrected notebook dropped: a `quick_analysis` driver over the whole folder, an emotion-flow
visualisation, and a `quick_csv_analysis` CSV exporter. They are restored here, **adapted to the
corrected pipeline**:

* They read the folder from **`CONFIG["data_folder"]`** (was a hard-coded `/content/data`) and use
  **`CONFIG["comparison_type"]`** for the KLD.
* They go through the corrected `analyze_tale_with_methods`, so the surprisal they report is the
  **smoothing-corrected KLD** (Fix 1), not the old inflated `1e-10` version.
* `analyze_tale_with_methods` now returns a **tuple** `(df, kld_scores)` and produces an `emo_kld` /
  `kld_score` column instead of `emotion_surprisal`. A thin adapter (`analyze_corpus`) unpacks the
  tuple and exposes an `emotion_surprisal` alias (= corrected `kld_score`, NaN→0.0) so the original
  `create_summary_statistics`, `plot_surprisal_heatmap` and `visualize_emotion_flow` work unchanged.

The corrected analyzer class itself is **not modified** — this section only adds driver code on top.

In [ ]:
# ===================== FULL-CORPUS DRIVER (Human Stories Only) =====================
def analyze_corpus(folder=None, comparison_type=None):
    """
    Modified driver to focus ONLY on human-written stories.
    Filters out AI-generated files (those containing '#_tt_gen_')
    and cleans story titles using the extract_story_title helper.
    """
    folder = folder or CONFIG["data_folder"]
    comparison_type = comparison_type or CONFIG["comparison_type"]

    frames = []
    for filename in sorted(os.listdir(folder)):
        # Skip non-text files and AI-generated files
        if not filename.endswith(".txt") or "_gen_" in filename:
            continue

        fpath = os.path.join(folder, filename)
        try:
            tale_df, _ = analyzer.analyze_tale_with_methods(
                fpath, segmentation_method="text_tiling", comparison_type=comparison_type)
        except Exception as e:
            print(f"  ! {filename}: {e}")
            continue

        if tale_df is None:
            continue

        # CLEANUP: Strip filename characters to get only the strictly clean title
        clean_title = extract_story_title(filename)

        tale_df = tale_df.copy()
        tale_df["tale"] = clean_title
        tale_df["emotion_surprisal"] = tale_df["emo_kld"]
        frames.append(tale_df)

    if not frames:
        print("No human-written tale files found!")
        return None
    return pd.concat(frames, ignore_index=True)

def quick_analysis(folder=None):
    """
    Quick full-corpus analysis for human stories only.
    Explicitly groups 'forgotten' tales at the bottom in bold.
    """
    folder = folder or CONFIG["data_folder"]
    print("Starting analysis of Human-Written Grimm's folktales...")

    df = analyze_corpus(folder)
    if df is None:
        return None

    # User-provided list of forgotten tales
    forgotten_tales = [
        "Jorinda And Jorindel", "The Crows And The Soldier", "The Faithful Beasts",
        "The Gold Children", "The Knapsack The Hat And The Horn",
        "The Little Brother And Sister", "The Nose",
        "The Sparrow And His Four Children", "The Stolen Farthings",
        "Three Little Tales About Toads"
    ]

    available_tales = df['tale'].unique()
    actual_forgotten = [t for t in forgotten_tales if t in available_tales]
    others = sorted([t for t in available_tales if t not in actual_forgotten])

    tales_sorted = others + actual_forgotten
    df['tale'] = pd.Categorical(df['tale'], categories=tales_sorted, ordered=True)
    df = df.sort_values(['tale', 'segment_id'])

    print(f"Analysis complete! Processed {df['tale'].nunique()} human tales.")

    # Display heatmap with bold highlighting for the grouped forgotten tales
    analyzer.plot_surprisal_heatmap(df, highlight_tales=actual_forgotten)

    # FIXED: Only fillna(0) on numeric columns to avoid Categorical type error
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df_filled = df.copy()
    df_filled[numeric_cols] = df_filled[numeric_cols].fillna(0)

    summary = analyzer.create_summary_statistics(df_filled)
    print("\nMost emotionally surprising segments (Human Only):")
    for segment in summary['most_surprising_segments']:
        print(f"- {segment['tale']}, Segment {segment['segment_id']}: {segment['emotion_surprisal']:.3f}")

    return df

print("Grimm's Folktales Human-Only Sentiment Analyzer (Fixed) is ready!")

In [ ]:
# Run the full-corpus analysis over CONFIG["data_folder"]
df = quick_analysis()

### 9.1 Emotion-flow visualisation for a single tale
`visualize_emotion_flow` (corrected class) plots the per-segment emotion distribution, the
VADER/TextBlob sentiment, and both surprisal measures (corrected KLD + rhombus) for one tale.

In [ ]:
# Visualise one tale's emotion flow. The original used 'Snow-White';
# we keep that default but fall back to the first available tale and list the options.
def show_emotion_flow(df, tale_name="Snow-White"):
    available = sorted(df["tale"].unique())
    if tale_name not in available:
        print(f"'{tale_name}' not found. Available tales:")
        for t in available:
            print("  -", t)
        tale_name = available[0]
        print(f"\nShowing '{tale_name}' instead.")
    analyzer.visualize_emotion_flow(df, tale_name)

show_emotion_flow(df, "Snow-White")

### 9.2 CSV summary export
`quick_csv_analysis` writes a one-row-per-tale summary (tokens, segments, average emotions and
sentiment) via the corrected `create_csv_summary`, using the comparison type from `CONFIG`.

In [ ]:
# ===================== CSV SUMMARY EXPORT (adapted from the original notebook) =====================
def quick_csv_analysis(folder=None, output_base="grimm_sentiment_analysis"):
    """
    Build a per-tale summary CSV (restored from the original notebook).
    Adapted: folder and comparison_type come from CONFIG instead of being hard-coded
    to '/content/AI_data/tt_start' and 'contextual'.
    """
    folder = folder or CONFIG["data_folder"]
    comparison_type = CONFIG["comparison_type"]
    print("Starting CSV analysis of Grimm's folktales...")

    summary_df = analyzer.create_csv_summary(
        folder,
        f"{output_base}_summary_{comparison_type}.csv",
        segmentation_method="text_tiling",
        comparison_type=comparison_type,
    )

    if summary_df is not None:
        print(f"\nSummary: Processed {len(summary_df)} tales")
        print("\nSample summary data:")
        print(summary_df[['Story', 'Characters', 'Tokens', 'Segments']].head().to_string(index=False))

    return summary_df

# Run it. (Original ran on '/content/AI_data/tt_start'; pass a folder to reproduce that.)
summary_df = quick_csv_analysis()

In [ ]:
# ============ SENTIMENT OVERVIEW TABLE (UPDATED) ============
import os, numpy as np, pandas as pd

# Using /content as base and filtering by filename markers
BASE_FOLDER = "/content"

def corpus_sentiment_means(folder, name_filter=None):
    per_tale = []
    if not os.path.exists(folder):
        return None

    for fn in sorted(os.listdir(folder)):
        if not fn.endswith(".txt"):
            continue
        if name_filter is not None and not name_filter(fn):
            continue

        try:
            df, _ = analyzer.analyze_tale_with_methods(
                os.path.join(folder, fn),
                segmentation_method="text_tiling")

            if df is not None and len(df) > 0:
                per_tale.append({
                    "Avg_VADER": df["vader_compound"].mean(),
                    "Avg_TextBlob": df["textblob_polarity"].mean(),
                    "Avg_Subjectivity": df["textblob_subjectivity"].mean(),
                })
        except Exception as e:
            continue

    if not per_tale:
        return None

    t = pd.DataFrame(per_tale)
    return {c: t[c].mean() for c in ["Avg_VADER", "Avg_TextBlob", "Avg_Subjectivity"]}

# --- Build the table using filters ---
results = {}

# 1. Human Tales (no "_gen_" marker)
hum_res = corpus_sentiment_means(BASE_FOLDER, name_filter=lambda f: "_gen_" not in f)
if hum_res: results["adj_tt_hum"] = hum_res

# 2. AI Tales - Text Tiling (markers "_tt_" and "_gen_")
ai_tt_res = corpus_sentiment_means(BASE_FOLDER, name_filter=lambda f: "_tt_" in f and "_gen_" in f)
if ai_tt_res: results["tt_st_adj_tt_gen"] = ai_tt_res

# 3. AI Tales - Equal Length (markers "_el_" and "_gen_")
ai_el_res = corpus_sentiment_means(BASE_FOLDER, name_filter=lambda f: "_el_gen_" in f)
if ai_el_res: results["adj_el_gen"] = ai_el_res

if not results:
    print("Error: No data found. Please ensure .txt files are in /content.")
else:
    tab = pd.DataFrame(results).T
    print("\nSentiment overview (recomputed with adj_el_gen):")
    print(tab.to_string(float_format=lambda x: f"{x:.9f}"))

    # LaTeX Output
    def latex_sentiment_table(tab):
        cols = ["Avg_VADER", "Avg_TextBlob", "Avg_Subjectivity"]
        mx = {c: tab[c].max() for c in cols}
        mn = {c: tab[c].min() for c in cols}
        def cell(c, v): return f"\\textbf{{{v:.9f}}}" if (v == mx[c] or v == mn[c]) else f"{v:.9f}"

        L = [r"\begin{table*}[h]", r"\centering", r"\caption{Sentiment Overview}", r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}llll}", r"\toprule", r"Config & Avg\_VADER & Avg\_TextBlob & Avg\_Subjectivity\\", r"\midrule"]
        for label in tab.index:
            L.append(f"{label.replace('_', r'\_')} & {cell('Avg_VADER', tab.loc[label,'Avg_VADER'])} & {cell('Avg_TextBlob', tab.loc[label,'Avg_TextBlob'])} & {cell('Avg_Subjectivity', tab.loc[label,'Avg_Subjectivity'])} \\\\")
        L += [r"\bottomrule", r"\end{tabular*}", r"\end{table*}"]
        return "\n".join(L)

    print("\n--- LaTeX ---\n")
    print(latex_sentiment_table(tab))

In [ ]:
print(f"Anzahl gefundener Ergebnisse: {len(results)}")
if results:
    temp_tab = pd.DataFrame(results).T
    print("Verfügbare Spalten im DataFrame:")
    print(temp_tab.columns.tolist())
    print("\nDataFrame Vorschau:")
    display(temp_tab.head())
else:
    print("Das 'results' Dictionary ist leer. Prüfe, ob .txt Dateien in /content liegen und die Filterkriterien erfüllen.")

In [ ]:
# ============ MOST EMOTIONALLY SURPRISING SEGMENTS (ALL CATEGORIES) ============
import os, numpy as np, pandas as pd

SURP_FOLDER = CONFIG["data_folder"]
TOP_N = 5

def get_top_surprising(name_filter, label):
    frames = []
    for fn in sorted(os.listdir(SURP_FOLDER)):
        if not fn.endswith(".txt") or not name_filter(fn):
            continue
        try:
            df, _ = analyzer.analyze_tale_with_methods(
                os.path.join(SURP_FOLDER, fn),
                segmentation_method="text_tiling",
                comparison_type=CONFIG["comparison_type"])
            if df is not None and len(df):
                df['tale'] = extract_story_title(fn)
                df['category'] = label
                frames.append(df)
        except:
            continue
    if not frames: return pd.DataFrame()
    corpus = pd.concat(frames, ignore_index=True)
    defined = corpus.dropna(subset=["emo_kld"]).copy()
    return defined

# Collect data for all three categories
hum_df = get_top_surprising(lambda f: "_gen_" not in f, "adj_tt_hum")
tt_gen_df = get_top_surprising(lambda f: "_tt_" in f and "_gen_" in f, "adj_tt_gen")
el_gen_df = get_top_surprising(lambda f: "_el_gen_" in f, "adj_el_gen")

all_data = pd.concat([hum_df, tt_gen_df, el_gen_df], ignore_index=True)

if all_data.empty:
    print("No data found.")
else:
    print("Top Surprising Segments by Category:")
    for cat in ["adj_tt_hum", "adj_tt_gen", "adj_el_gen"]:
        subset = all_data[all_data['category'] == cat]
        if not subset.empty:
            print(f"\n--- {cat} ---")
            print(subset.nlargest(TOP_N, "emo_kld")[["tale", "segment_id", "emo_kld"]].to_string(index=False))

    # Generate Summary Table
    summary = all_data.groupby('category').agg({
        'emo_kld': ['mean', 'max'],
        'vader_compound': 'mean'
    })
    summary.columns = ['Avg_Surprisal', 'Max_Surprisal', 'Avg_VADER']
    print("\nSummary Statistics:")
    print(summary.to_string(float_format=lambda x: f"{x:.4f}"))